In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
#Paths to all of the useful supplemental tables and tensors
active_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor.hdf5"
passive_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbnAllUnitSpikeTensor_passive.hdf5"
opto_tensor_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/vbn_opto_tensor_unit_grouped.hdf5"

stim_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_stim_table_no_filter.csv"
unit_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_units_with_responsiveness.csv"
unit_table_with_rf_stats = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/units_with_rf_stats.csv"
unit_table_opto_metrics = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/unit_opto_metrics.csv"
unit_cluster_labels = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/unit_cluster_labels.csv"
unit_vis_responsive = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/units_with_vis_responsiveness.csv"

sessions_table_file = "/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/supplemental_tables/master_sessions_table.csv"

In [ ]:
sessions = pd.read_csv(sessions_table_file)

In [ ]:
surgery_info = pd.read_excel("/Volumes/programs/mindscope/workgroups/np-behavior/VBN Manuscript/2019-2022 SWR list VBN.xlsx")

In [ ]:
mice = sessions.groupby('mouse_id').head(1)

In [ ]:
mouse_surgery = mice.merge(surgery_info[['LabTracks ID', 'Date of HP Surgery', 'HP Surgeon']], left_on='mouse_id', right_on='LabTracks ID', how='left')

In [ ]:
mouse_surgery['Date of HP Surgery']

In [ ]:
import sys
sys.path.append(r"C:\Users\svc_ccg\Documents\GitHub\NP_pipeline_QC")

In [ ]:
import query_lims

In [ ]:
mouse_data = query_lims.query_lims(query_lims.DONOR_FROM_LABTRACKS_QRY.format(431664))
mouse_data[0]['date_of_birth']

In [ ]:
mouse_dobs = {}
for mouse in mouse_surgery['mouse_id'].unique():
    try:
        mouse_data = query_lims.query_lims(query_lims.DONOR_FROM_LABTRACKS_QRY.format(mouse))
        mouse_dobs[mouse] = mouse_data[0]['date_of_birth']
    except Exception as e:
        print(f"Error retrieving DOB for {mouse}: {e}")
    


In [ ]:
mouse_dobs_df = pd.DataFrame.from_dict(mouse_dobs, orient='index', columns=['date_of_birth']).reset_index()

mouse_surgery = mouse_surgery.merge(mouse_dobs_df, left_on='mouse_id', right_on='index', how='left')

In [ ]:
mouse_surgery['age_at_surgery'] = mouse_surgery.apply(lambda row: row['Date of HP Surgery'] - row['date_of_birth'], axis=1)

In [ ]:
mouse_surgery['age_at_surgery'].describe()

In [ ]:
mouse_surgery.iloc[0]

In [ ]:
stims = pd.read_csv(stim_table_file)

In [ ]:
stims[stims['is_change']]['trial_flash'].value_counts()

In [ ]:
def get_block_durations(group):
    
    block_starts = group.groupby('stimulus_block').head(1)['start_time']
    block_ends = group.groupby('stimulus_block').tail(1)['stop_time']
    durations = block_ends.values - block_starts.values
    return durations


In [ ]:
stims.groupby('ecephys_session_id').apply(lambda g: get_block_durations(g))

In [ ]:
sum(sessions['abnormal_histology'].notnull()), sum(sessions['abnormal_activity'].notnull()), sum(sessions['abnormal_histology'].notnull()&sessions['abnormal_activity'].notnull()), 

In [ ]:
units = pd.read_csv(unit_table_file)

In [ ]:
units_by_probe = units.groupby('ecephys_probe_id').apply(lambda group: len(np.unique(group['structure_acronym'])))
len(units_by_probe[units_by_probe<3])

In [ ]:
units[units['ecephys_probe_id'].isin(units_by_probe[units_by_probe<4].index.values)].value_counts('structure_acronym')

In [ ]:
units[units['ecephys_probe_id'].isin(units[units['structure_acronym']=='grey']['ecephys_probe_id'].unique())].value_counts('structure_acronym')

In [ ]:
units['structure_acronym'].value_counts().sort_values(ascending=False)[:50]

In [ ]:
units.groupby('ecephys_session_id')

In [ ]:
probes = pd.read_csv("/Volumes/programs/mindscope/workgroups/np-behavior/vbn_data_release/vbn_s3_cache/visual-behavior-neuropixels-0.5.0/project_metadata/probes.csv")

In [ ]:
len(probes[(probes['structure_acronyms'].str.contains('grey')) & (probes['structure_acronyms'].str.count(',')==2)]), len(probes)

In [ ]:
lost_probes = probes[(probes['structure_acronyms'].str.contains('grey')) & (probes['structure_acronyms'].str.count(',')==2)]['ecephys_probe_id'].unique()

In [ ]:
from decoding_utils import apply_condition_filter
import decoding_utils as du

In [ ]:
units['ecephys_session_id'].nunique()

In [ ]:
quality_filter = du.apply_unit_quality_filter(units)
noabnorm_units = units.loc[quality_filter]

ggh_filter = np.logical_or(apply_condition_filter(noabnorm_units, 'Familiar', 'GGH'), apply_condition_filter(noabnorm_units, 'Novel', 'GGH'))
ghg_filter = np.logical_or(apply_condition_filter(noabnorm_units, 'Familiar', 'GHG'), apply_condition_filter(noabnorm_units, 'Novel', 'GHG'))
hhg_filter = np.logical_or(apply_condition_filter(noabnorm_units, 'Familiar', 'HHG'), apply_condition_filter(noabnorm_units, 'Novel', 'HHG'))

print(noabnorm_units.loc[ggh_filter]['mouse_id'].nunique(), noabnorm_units.loc[ghg_filter]['mouse_id'].nunique(), noabnorm_units.loc[hhg_filter]['mouse_id'].nunique(), noabnorm_units['mouse_id'].nunique())

## comparing to VBO strategy data

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
vbo_beh_model_summary = pd.read_pickle("/Volumes/programs/braintv/workgroups/nc-ophys/alex.piet/behavior/psy_fits_v21/summary_data/_summary_table.pkl")
vbn_beh_model_summary = pd.read_pickle("/Volumes/programs/braintv/workgroups/nc-ophys/alex.piet/NP/behavior/psy_fits_v100/summary_data/_summary_table.pkl")

In [ ]:
for c in vbn_beh_model_summary.columns:
    print(c)

In [ ]:
print('timing', np.sum(vbo_beh_model_summary['strategy_dropout_index']<0)/len(vbo_beh_model_summary), np.sum(vbn_beh_model_summary['strategy_dropout_index']<0)/len(vbn_beh_model_summary))
print('visual', np.sum(vbo_beh_model_summary['strategy_dropout_index']>0)/len(vbo_beh_model_summary), np.sum(vbn_beh_model_summary['strategy_dropout_index']>0)/len(vbn_beh_model_summary))

In [ ]:
vbn_beh_model_summary['strategy_dropout_index'].hist(density=True, bins=25, alpha=0.5, label='VBN')
vbo_beh_model_summary['strategy_dropout_index'].hist(density=True, bins=25, alpha=0.5, label='VBO')

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
# Assuming the DataFrame is named `vbo_beh_model_summary`
grouped = vbo_beh_model_summary.groupby('mouse_id')['strategy_dropout_index'].mean().sort_values(ascending=False)

fig, ax = plt.subplots()
for im, mouse in enumerate(grouped.index[::-1]):
    mouse_data = vbo_beh_model_summary[vbo_beh_model_summary['mouse_id'] == mouse]
    ax.plot([im]*len(mouse_data), mouse_data['strategy_dropout_index'], 'o', alpha=0.5,color='black', markersize=2)

grouped = vbn_beh_model_summary.groupby('mouse_id')['strategy_dropout_index'].mean().sort_values(ascending=False)

for im, mouse in enumerate(grouped.index[::-1]):
    mouse_data = vbn_beh_model_summary[vbn_beh_model_summary['mouse_id'] == mouse]
    ax.plot([im]*len(mouse_data), mouse_data['strategy_dropout_index'], 'o', alpha=0.5,color='red', markersize=2)

In [ ]:
import vbn_utils as vbn
fig, ax = plt.subplots()
plt.rcParams['font.size'] = 18
for id, dataset in enumerate([vbn_beh_model_summary, vbo_beh_model_summary]):
    plt.boxplot(dataset['strategy_dropout_index'], positions=[id], widths=0.5, patch_artist=True, boxprops=dict(facecolor='gray', color='black'), 
                medianprops=dict(color='black'), whiskerprops=dict(color='black'), capprops=dict(color='black'), notch=True, showfliers=False)
vbn.formatFigure(fig, ax, xLabel='Dataset', yLabel='Strategy dropout index')
ax.set_xticklabels(['VBN', 'VBO'])
ax.axhline(0, color='k', ls='dotted')

In [ ]:
import scipy.stats

vbn_beh_model_summary['strategy_dropout_index'].mean(), vbn_beh_model_summary['strategy_dropout_index'].sem(), vbo_beh_model_summary['strategy_dropout_index'].mean(), vbo_beh_model_summary['strategy_dropout_index'].sem(), scipy.stats.ranksums(vbn_beh_model_summary['strategy_dropout_index'], vbo_beh_model_summary['strategy_dropout_index'])

## for author list

In [ ]:
surgery_counts = pd.DataFrame(mouse_surgery['HP Surgeon'].value_counts())
surgery_counts

In [ ]:
trainer_behavior_sessions_for_mouse_id_query = '''
    SELECT u.login
    FROM behavior_sessions bs
    JOIN users as u
    ON u.id = bs.user_id
    JOIN donors AS d
    ON bs.donor_id = d.id
    WHERE d.external_donor_name=cast({} as character varying)
    '''

In [ ]:
trainers = {m:[] for m in mouse_surgery['LabTracks ID'].unique()}
four_day_mice = ['729293', '725107', '725108', '729286', '729287']
all_trainers = []
all_mice = []
for mouse in list(mouse_surgery['LabTracks ID'].unique()) + four_day_mice:
    try:
        bs = query_lims.query_lims(trainer_behavior_sessions_for_mouse_id_query.format(mouse))
        if len(bs) > 0:
            trainers[mouse] = [b['login'] for b in bs]
            all_trainers.extend([b['login'] for b in bs])
            all_mice.extend([mouse] * len(bs))
        else:
            print(f"Mouse {mouse} has no trainer sessions.")
    except Exception as e:
        print(f"Error retrieving trainer sessions for mouse {mouse}: {e}")

all_trainers = np.array(all_trainers)
all_mice = np.array(all_mice)

In [ ]:
trainer_df = pd.DataFrame({'trainer': all_trainers, 'mouse_id': all_mice})

In [ ]:
trainer_df.value_counts('trainer')

In [ ]:
isi_query = '''
SELECT u.login
from isi_experiments isi
JOIN specimens sp
ON sp.id = isi.specimen_id
JOIN users u
ON u.id = isi.operator_id
WHERE sp.external_specimen_name = cast({} as character varying)
'''

In [ ]:
all_isi_operators = []
all_isi_mice = []
for mouse in mouse_surgery['LabTracks ID'].unique():
    try:
        isi = query_lims.query_lims(isi_query.format(mouse))
        if len(isi) > 0:
            all_isi_operators.extend([b['login'] for b in isi])
            all_isi_mice.extend([mouse] * len(isi))
        else:
            print(f"Mouse {mouse} has no trainer sessions.")
    except Exception as e:
        print(f"Error retrieving trainer sessions for mouse {mouse}: {e}")

all_isi_operators = np.array(all_isi_operators)
all_isi_mice = np.array(all_isi_mice)


In [ ]:
isi_df = pd.DataFrame({'isi_operator': all_isi_operators, 'mouse_id': all_isi_mice})

In [ ]:
login_to_name = {
    'jackies': 'Jackie Swapp',
    'tye.johnson': 'Tye Johnson',
    'conor.grasso': 'Conor Grasso',
    'josh.wilkes': 'Josh Wilkes',
    'hannah.belski': 'Hannah Belski',
    'beno': 'Ben Ouellette',
    'taminar' : 'Tamina Ramirez',
    'chelsean': 'Chelsea Nayan',
    'greggh' : 'Gregg Heller',
    'katn' : 'Kat North',
    'shiellac' : 'Shiella Caldejon',
    'paul.rhoads': 'Paul Rhoads',
    'xana.waughman': 'Xana Waughman',
    'india.kato': 'India Kato',
    'sara.kivikas': 'Sara Kivikas',
    'vivian.ha': 'Vivian Ha',
    'henry.loeffler': 'Henry Loeffler',
}

In [ ]:
isi_df['name'] = isi_df.apply(lambda row: login_to_name.get(row['isi_operator'], row['isi_operator']), axis=1)
isi_counts = pd.DataFrame(isi_df.value_counts('name').sort_values(ascending=False))

In [ ]:
trainer_df['name'] = trainer_df.apply(lambda row: login_to_name.get(row['trainer'], row['trainer']), axis=1)
behavior_session_counts = pd.DataFrame(trainer_df.value_counts('name').sort_values(ascending=False))

In [ ]:
behavior_isi = behavior_session_counts.merge(isi_counts, left_index=True, right_index=True, how='outer', suffixes=('_behavior', '_isi')).fillna(0).astype(int).sort_values(by='name')
behavior_isi_surgery = behavior_isi.merge(surgery_counts,left_index=True, right_index=True, how='outer', suffixes=('_behavior', '_isi')).fillna(0).astype(int)
bis_abridged = behavior_isi_surgery.iloc[:-4].rename(columns={'0_behavior': 'behavior_sessions', '0_isi': 'isi_sessions', 'HP Surgeon': 'surgeries'})
bis_abridged.sort_index()

In [ ]:
def highlight_rows(row):
    if row['behavior_sessions'] >= 100: 
        return ['background-color: green'] * len(row)
    if row['isi_sessions'] >= 10:  # Criteria: Score < 80
        return ['background-color: green'] * len(row)
    if row['surgeries']>5:
        return ['background-color: green'] * len(row)
    
    else:  # Default style
        return [''] * len(row)

In [ ]:
bis_abridged.style.apply(highlight_rows, axis=1)

In [ ]:
#list qualifying authors
qualifying_authors = bis_abridged[(bis_abridged['behavior_sessions'] >= 100) | (bis_abridged['isi_sessions'] >= 10) | (bis_abridged['surgeries'] > 5)].index.tolist()
for qa in qualifying_authors:
    print(qa + '\n')


In [ ]:
#For TCM colony management
mouse_surgery.sort_values(by='date_of_birth').groupby('genotype').first()[['mouse_id', 'date_of_birth']], mouse_surgery.sort_values(by='date_of_birth').groupby('genotype').last()[['mouse_id', 'date_of_birth']]

In [ ]:
for genotype, group in mouse_surgery.sort_values(by='date_of_birth').groupby('genotype'):
    plt.figure()
    plt.title(genotype)
    plt.hist(group['date_of_birth'], alpha=0.5, label=genotype)


In [ ]:
mouse_surgery.sort_values(by='date_of_birth').groupby('genotype').apply(lambda x: plt.hist(x['date_of_birth'], alpha=0.5, bins=12, label=x['genotype'].iloc[0]))
plt.legend()

## Author contribution matrix

In [ ]:
author_df = pd.read_excel("/Volumes/programs/mindscope/workgroups/np-behavior/VBN Manuscript/VBN_authors.xlsx")

In [ ]:
author_df.sort_values(by=['Author Last', 'Author First'], inplace=True)

In [ ]:
alpha = author_df.sort_values(by=['Author Last', 'Author First'])

In [ ]:
just_contributions = alpha.iloc[:, [2] + list(range(7, alpha.shape[1]))]

In [ ]:
# Create a matrix visualization of just_contributions
fig, ax = plt.subplots(figsize=(12, 14))

# Create the matrix plot using imshow
im = ax.imshow(just_contributions.iloc[:, 1:].values, cmap='Greys', aspect='equal')

# Set row labels (full names)
ax.set_yticks(range(len(just_contributions)))
ax.set_yticklabels(just_contributions.iloc[:, 0])

# Set column labels at the top
ax.set_xticks(range(len(just_contributions.columns[1:])))
ax.set_xticklabels(just_contributions.columns[1:], rotation=45, ha='left')
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')

# Add title
# ax.set_title('Author Contribution Matrix', pad=20)

# Adjust layout to prevent label cutoff
plt.tight_layout()
plt.show()


In [ ]:
# Create a condensed text output showing contributors for each column
print("VBN Author Contribution Summary")
print("=" * 50)

for col in just_contributions.columns[1:]:
    contributors = just_contributions[just_contributions[col] == 1]['Full Name'].tolist()
    print(f"\n{col}:")
    if contributors:
        print(", ".join(contributors))
    else:
        print("No contributors")

# Create another version using initials instead of full names
print("\n" + "=" * 50)
print("VBN Author Contribution Summary (Initials)")
print("=" * 50)

for col in just_contributions.columns[1:]:
    contributors = just_contributions[just_contributions[col] == 1]['Full Name'].tolist()
    print(f"\n{col}:")
    if contributors:
        # Convert full names to initials
        initials = []
        for name in contributors:
            parts = name.split()
            if len(parts) >= 2:
                initial = parts[0][0] + '.' + parts[-1][0] + '.'  # First name initial + Last name initial
                initials.append(initial)
            else:
                initials.append(name[0] if name else "")  # Just first character if single name
        print(", ".join(initials))
    else:
        print("No contributors")


In [ ]:
import pandas as pd

d = pd.read_excel("/Volumes/programs/mindscope/workgroups/dynamicrouting/DynamicRoutingTask/DynamicRoutingTraining.xlsx", sheet_name='all mice')

In [ ]:
mice_in_training = d[d['status']=='training']
mice_in_training